# Phase 1 - Layer A: Starlink Satellite Orbital Data

This notebook builds the first realistic data layer for the scheduling project: a clean, reproducible subset of real Starlink orbital records.

CelesTrak publishes General Perturbations (GP) data in Two-Line Element (TLE) format. A TLE is a compact text representation of a satellite orbit at a particular epoch. Tools such as `skyfield` use this format with the SGP4 orbit model to propagate satellite positions over time.

Starlink is used here because it is public, large enough to create realistic congestion later, and directly supported by the CelesTrak Starlink group endpoint. The output of this notebook is a validated satellite table containing names, NORAD catalog IDs, source metadata, TLE lines, and epochs for a configurable subset of satellites.

## Configuration

Adjust `SATELLITE_SUBSET_SIZE` to change how many Starlink satellites are kept for downstream experiments. The notebook downloads from CelesTrak when possible, stores a local cache in `starlink-satellites-data`, and writes the cleaned subset back to that same Layer A folder.

The first code cell also checks whether `pandas` and `skyfield` are installed in the active notebook kernel. If a package is missing, it installs it with the same Python executable that is running the notebook, which avoids the common mismatch where terminal `pip` installs into a different environment.

In [ ]:
from pathlib import Path
from urllib.request import urlopen, Request
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = ["pandas", "skyfield"]
missing_packages = [
    package_name
    for package_name in REQUIRED_PACKAGES
    if importlib.util.find_spec(package_name) is None
]

if missing_packages:
    print(f"Installing missing packages into this notebook kernel: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print("Dependency installation finished. If imports still fail, restart the kernel and run the notebook again.")

import pandas as pd
from skyfield.api import EarthSatellite, load

CELESTRAK_STARLINK_TLE_URL = "https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle"
SATELLITE_SUBSET_SIZE = 100
REFRESH_TLE_CACHE = False

DATA_DIR = Path("../../data/dataset/starlink")
DATA_DIR.mkdir(parents=True, exist_ok=True)
TLE_CACHE_PATH = DATA_DIR / "starlink_celestrak_latest.tle"
OUTPUT_CSV_PATH = DATA_DIR / "starlink_satellite_subset.csv"

print(f"Notebook Python executable: {sys.executable}")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"TLE cache path: {TLE_CACHE_PATH.resolve()}")
print(f"Output CSV path: {OUTPUT_CSV_PATH.resolve()}")

Notebook Python executable: d:\University\PoliMi\courses\SWE2\SWE2-research-project\.venv\Scripts\python.exe
Data directory: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink_satellites
TLE cache path: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink_satellites\starlink_celestrak_latest.tle
Output CSV path: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink_satellites\starlink_satellite_subset.csv


## Load Starlink TLE Data

The loader prefers reproducibility: it reuses the local cache unless `REFRESH_TLE_CACHE` is set to `True`. If a download fails but a cache already exists, the cached file is used so that review and reruns still work offline.

In [19]:
def load_starlink_tle_text(url: str, cache_path: Path, refresh_cache: bool = False) -> str:
    """Download Starlink TLE text from CelesTrak or load a local cached copy.

    Parameters
    ----------
    url:
        CelesTrak endpoint for the Starlink TLE group.
    cache_path:
        Local path used to store and reuse the downloaded TLE text.
    refresh_cache:
        When True, force a new download before reading the data.

    Returns
    -------
    str
        Raw TLE text containing repeating name, line 1, and line 2 records.
    """
    if cache_path.exists() and not refresh_cache:
        return cache_path.read_text(encoding="utf-8")

    try:
        req = Request(url, headers={
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            "Accept": "text/plain,*/*"
        }) # trying to download the stuff
        with urlopen(req, timeout=30) as response:
            text = response.read().decode("utf-8")
        cache_path.write_text(text, encoding="utf-8")
        return text
    except Exception as exc:
        if cache_path.exists():
            print(f"Download failed ({exc}); using existing local cache.")
            return cache_path.read_text(encoding="utf-8")
        raise RuntimeError("Could not download Starlink TLE data and no local cache exists.") from exc


tle_text = load_starlink_tle_text(
    CELESTRAK_STARLINK_TLE_URL,
    TLE_CACHE_PATH,
    refresh_cache=REFRESH_TLE_CACHE,
)

print(f"Loaded {len(tle_text.splitlines())} raw TLE text lines.")

Loaded 62370 raw TLE text lines.


## Parse and Clean Satellite Records

Each Starlink satellite record is expected to have three lines: the satellite name, TLE line 1, and TLE line 2. `skyfield` is used to parse each record and expose the propagated object metadata, including the TLE epoch.

In [20]:
def parse_starlink_tles(tle_text: str, source: str) -> pd.DataFrame:
    """Parse Starlink TLE text into a clean satellite table.

    Parameters
    ----------
    tle_text:
        Raw text in three-line TLE format: satellite name, line 1, and line 2.
    source:
        Human-readable provenance label saved with each satellite row.

    Returns
    -------
    pandas.DataFrame
        Table with satellite name, NORAD ID, source, raw TLE lines, and epoch.
    """
    lines = [line.strip() for line in tle_text.splitlines() if line.strip()]
    if len(lines) % 3 != 0:
        raise ValueError(f"Expected TLE text in 3-line records, found {len(lines)} non-empty lines.")

    timescale = load.timescale()
    records = []

    for index in range(0, len(lines), 3):
        satellite_name = lines[index]
        tle_line_1 = lines[index + 1]
        tle_line_2 = lines[index + 2]

        satellite = EarthSatellite(tle_line_1, tle_line_2, satellite_name, timescale)
        records.append(
            {
                "satellite_name": satellite.name,
                "norad_id": satellite.model.satnum,
                "source": source,
                "tle_line_1": tle_line_1,
                "tle_line_2": tle_line_2,
                "epoch": satellite.epoch.utc_iso(),
            }
        )

    return pd.DataFrame.from_records(records)


satellites = parse_starlink_tles(
    tle_text,
    source="CelesTrak Starlink GP/TLE",
)

satellites.head()

,satellite_name,norad_id,source,tle_line_1,tle_line_2,epoch
0,STARLINK-1008,44714,CelesTrak Starlink GP/TLE,1 44714U 19074B 26151.21337775 .00088051 0...,2 44714 53.1512 149.8655 0002987 57.7638 302...,2026-05-31T05:07:16Z
1,STARLINK-1012,44718,CelesTrak Starlink GP/TLE,1 44718U 19074F 26151.21269194 .00096054 0...,2 44718 53.1557 150.1665 0002806 53.9952 306...,2026-05-31T05:06:17Z
2,STARLINK-1017,44723,CelesTrak Starlink GP/TLE,1 44723U 19074L 26150.51919821 .00042312 0...,2 44723 53.0464 151.8820 0005612 11.2177 348...,2026-05-30T12:27:39Z
3,STARLINK-1020,44725,CelesTrak Starlink GP/TLE,1 44725U 19074N 26151.06717389 .00050090 0...,2 44725 53.1578 175.2046 0004536 32.1436 327...,2026-05-31T01:36:44Z
4,STARLINK-1036,44741,CelesTrak Starlink GP/TLE,1 44741U 19074AE 26151.20935272 .00077989 0...,2 44741 53.0413 144.5484 0003103 354.2287 5...,2026-05-31T05:01:28Z


## Select a Reproducible Experiment Subset

The subset is deterministic: after sorting by NORAD catalog ID, the first `SATELLITE_SUBSET_SIZE` records are kept. This keeps early experiments small while preserving a stable selection across reruns using the same source snapshot.

In [21]:
def select_satellite_subset(satellites_df: pd.DataFrame, subset_size: int) -> pd.DataFrame:
    """Return a deterministic subset of satellites for reviewable experiments.

    Parameters
    ----------
    satellites_df:
        Full parsed Starlink satellite table.
    subset_size:
        Maximum number of satellites to keep.

    Returns
    -------
    pandas.DataFrame
        Sorted and reindexed satellite subset.
    """
    if subset_size <= 0:
        raise ValueError("subset_size must be a positive integer.")

    return (
        satellites_df.sort_values(["norad_id", "satellite_name"])
        .head(subset_size)
        .reset_index(drop=True)
    )


satellite_subset = select_satellite_subset(satellites, SATELLITE_SUBSET_SIZE)

print(f"Loaded Starlink satellites: {len(satellites)}")
print(f"Kept in configured subset: {len(satellite_subset)}")
satellite_subset.head(10)

Loaded Starlink satellites: 10395
Kept in configured subset: 100


,satellite_name,norad_id,source,tle_line_1,tle_line_2,epoch
0,STARLINK-1008,44714,CelesTrak Starlink GP/TLE,1 44714U 19074B 26151.21337775 .00088051 0...,2 44714 53.1512 149.8655 0002987 57.7638 302...,2026-05-31T05:07:16Z
1,STARLINK-1012,44718,CelesTrak Starlink GP/TLE,1 44718U 19074F 26151.21269194 .00096054 0...,2 44718 53.1557 150.1665 0002806 53.9952 306...,2026-05-31T05:06:17Z
2,STARLINK-1017,44723,CelesTrak Starlink GP/TLE,1 44723U 19074L 26150.51919821 .00042312 0...,2 44723 53.0464 151.8820 0005612 11.2177 348...,2026-05-30T12:27:39Z
3,STARLINK-1020,44725,CelesTrak Starlink GP/TLE,1 44725U 19074N 26151.06717389 .00050090 0...,2 44725 53.1578 175.2046 0004536 32.1436 327...,2026-05-31T01:36:44Z
4,STARLINK-1036,44741,CelesTrak Starlink GP/TLE,1 44741U 19074AE 26151.20935272 .00077989 0...,2 44741 53.0413 144.5484 0003103 354.2287 5...,2026-05-31T05:01:28Z
5,STARLINK-1039,44744,CelesTrak Starlink GP/TLE,1 44744U 19074AH 26150.40294257 .00039408 0...,2 44744 53.0589 197.6402 0003669 15.3719 344...,2026-05-30T09:40:14Z
6,STARLINK-1042,44747,CelesTrak Starlink GP/TLE,1 44747U 19074AL 26150.95715872 .00030709 0...,2 44747 53.0431 58.8433 0003442 77.0901 283...,2026-05-30T22:58:19Z
7,STARLINK-1043,44748,CelesTrak Starlink GP/TLE,1 44748U 19074AM 26151.21230837 .00051622 0...,2 44748 53.0474 154.4197 0001066 68.5301 291...,2026-05-31T05:05:43Z
8,STARLINK-1046,44751,CelesTrak Starlink GP/TLE,1 44751U 19074AQ 26151.07866424 .00034522 0...,2 44751 53.0462 176.2260 0007338 30.0358 330...,2026-05-31T01:53:17Z
9,STARLINK-1047,44752,CelesTrak Starlink GP/TLE,1 44752U 19074AR 26151.20243433 .00016787 0...,2 44752 53.0674 120.0099 0003768 92.2811 267...,2026-05-31T04:51:30Z


## Validation

These checks make sure the Phase 1 output is usable by later propagation code: NORAD IDs must exist, TLE fields must be non-empty, and the subset should not contain duplicate catalog IDs.

In [22]:
def validate_satellite_table(satellites_df: pd.DataFrame) -> dict:
    """Validate the cleaned satellite table and return a compact report.

    Parameters
    ----------
    satellites_df:
        Satellite table to validate.

    Returns
    -------
    dict
        Counts for missing values, duplicate IDs, and table size.
    """
    required_columns = ["satellite_name", "norad_id", "source", "tle_line_1", "tle_line_2", "epoch"]
    missing_columns = [column for column in required_columns if column not in satellites_df.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    report = {
        "rows": len(satellites_df),
        "missing_norad_ids": int(satellites_df["norad_id"].isna().sum()),
        "empty_tle_line_1": int(satellites_df["tle_line_1"].fillna("").str.strip().eq("").sum()),
        "empty_tle_line_2": int(satellites_df["tle_line_2"].fillna("").str.strip().eq("").sum()),
        "duplicate_norad_ids": int(satellites_df["norad_id"].duplicated().sum()),
    }

    failed_checks = {name: value for name, value in report.items() if name != "rows" and value != 0}
    if failed_checks:
        raise ValueError(f"Satellite table validation failed: {failed_checks}")

    return report


validation_report = validate_satellite_table(satellite_subset)
validation_report

{'rows': 100,
 'missing_norad_ids': 0,
 'empty_tle_line_1': 0,
 'empty_tle_line_2': 0,
 'duplicate_norad_ids': 0}

## Save Phase 1 Output

The cleaned satellite subset is saved as a CSV because it is simple to inspect and easy for later notebooks to load. The raw TLE cache is also kept in this folder for reproducibility.

In [23]:
satellite_subset.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"Saved clean Starlink satellite subset to: {OUTPUT_CSV_PATH.resolve()}")
print(f"Rows saved: {len(satellite_subset)}")
satellite_subset.head(10)

Saved clean Starlink satellite subset to: D:\University\PoliMi\courses\SWE2\SWE2-research-project\data\dataset\starlink_satellites\starlink_satellite_subset.csv
Rows saved: 100


,satellite_name,norad_id,source,tle_line_1,tle_line_2,epoch
0,STARLINK-1008,44714,CelesTrak Starlink GP/TLE,1 44714U 19074B 26151.21337775 .00088051 0...,2 44714 53.1512 149.8655 0002987 57.7638 302...,2026-05-31T05:07:16Z
1,STARLINK-1012,44718,CelesTrak Starlink GP/TLE,1 44718U 19074F 26151.21269194 .00096054 0...,2 44718 53.1557 150.1665 0002806 53.9952 306...,2026-05-31T05:06:17Z
2,STARLINK-1017,44723,CelesTrak Starlink GP/TLE,1 44723U 19074L 26150.51919821 .00042312 0...,2 44723 53.0464 151.8820 0005612 11.2177 348...,2026-05-30T12:27:39Z
3,STARLINK-1020,44725,CelesTrak Starlink GP/TLE,1 44725U 19074N 26151.06717389 .00050090 0...,2 44725 53.1578 175.2046 0004536 32.1436 327...,2026-05-31T01:36:44Z
4,STARLINK-1036,44741,CelesTrak Starlink GP/TLE,1 44741U 19074AE 26151.20935272 .00077989 0...,2 44741 53.0413 144.5484 0003103 354.2287 5...,2026-05-31T05:01:28Z
5,STARLINK-1039,44744,CelesTrak Starlink GP/TLE,1 44744U 19074AH 26150.40294257 .00039408 0...,2 44744 53.0589 197.6402 0003669 15.3719 344...,2026-05-30T09:40:14Z
6,STARLINK-1042,44747,CelesTrak Starlink GP/TLE,1 44747U 19074AL 26150.95715872 .00030709 0...,2 44747 53.0431 58.8433 0003442 77.0901 283...,2026-05-30T22:58:19Z
7,STARLINK-1043,44748,CelesTrak Starlink GP/TLE,1 44748U 19074AM 26151.21230837 .00051622 0...,2 44748 53.0474 154.4197 0001066 68.5301 291...,2026-05-31T05:05:43Z
8,STARLINK-1046,44751,CelesTrak Starlink GP/TLE,1 44751U 19074AQ 26151.07866424 .00034522 0...,2 44751 53.0462 176.2260 0007338 30.0358 330...,2026-05-31T01:53:17Z
9,STARLINK-1047,44752,CelesTrak Starlink GP/TLE,1 44752U 19074AR 26151.20243433 .00016787 0...,2 44752 53.0674 120.0099 0003768 92.2811 267...,2026-05-31T04:51:30Z
